# Notebook BCUL - Alias and first ingestion

## Imports

In [1]:

# -*- coding: utf-8 -*-
import json
import os
import requests
import pandas as pd
import numpy as np
import certifi
import string
import unicodedata
from datetime import datetime, date, timedelta
from bs4 import BeautifulSoup
import unicodedata
import re
import pylcs
import sys

In [2]:
sys.path

['/rcp-scratch/iccluster040_scratch/students/banuls/.conda/envs/impresso_env/lib/python311.zip',
 '/rcp-scratch/iccluster040_scratch/students/banuls/.conda/envs/impresso_env/lib/python3.11',
 '/rcp-scratch/iccluster040_scratch/students/banuls/.conda/envs/impresso_env/lib/python3.11/lib-dynload',
 '',
 '/rcp-scratch/iccluster040_scratch/students/banuls/.conda/envs/impresso_env/lib/python3.11/site-packages']

In [4]:
from text_preparation.importers.bcul.detect import detect_issues, BculIssueDir,dir2issue
from text_preparation.importers.bcul.helpers import parse_date, find_mit_file # replace_alias does not exist anymore

In [5]:
from impresso_essentials.utils import ALL_MEDIA

## Variables

In [6]:
PATH_FILE_SYSTEM = '/mnt/project_impresso/original/BCUL/'

## How to Access the file-system

In [7]:
# All journals in the file-system
os.listdir(PATH_FILE_SYSTEM)

['Bulletins_du_Grand_Conseil',
 'Repertoires_de_Bulletins_du_Grand_Conseil',
 'Le_Bistouri',
 'Journal_de_Cossonay',
 'Aux_quatre_coins_du_MontMont_Journal',
 'Journal_de_Payerne',
 "L'Estafette_Journal_suisse",
 'Le_Charivari_[2]',
 'Arretez-les_tous_!',
 'Allez_savoir!',
 'Nouvelle_revue_de_Lausanne_et_du_Pays_de_Vaud',
 'Journal_du_Chablais',
 'La_Vague',
 "Feuille_d'avis_de_Morges_1",
 'Echo_du_Petit-LacRive_droite_Rive_gauche',
 'Journal_de_Vevey',
 "Feuille_d'avis_de_Biere_1",
 'Le_Semeur',
 'Messager_boîteux',
 'Presse_Lausannoise',
 'Pour_la_famille',
 "Le_Progres_de_Chateau_d'Oex",
 'Le_Touriste__La_Suisse_illustrée',
 'Journal_de_Bussigny',
 'Echo_de_la_Montagne',
 'La_Bombe',
 'La_Mouette',
 'Le_Frondeur',
 'Revue_photo-rigolotipique',
 'Nouvelle_revue_hebdo',
 'Le_Phare_du_Leman',
 "L'Echo_de_la_Broye",
 "L'Avenir_de_Vevey",
 'La_Veveysanne__La_Patrie',
 '.DS_Store',
 "L'Etincelle",
 'Croquis_vaudois',
 'Les_Alpes_vaudoises',
 'Journal_de_Moudon',
 'Le_Courrier_de_Prilly',


In [8]:
path = PATH_FILE_SYSTEM + 'Mercure_suisse'

## Functions

In [9]:
def normalize_title(title: str) -> str:
    # 1. Remove accents
    nfkd_form = unicodedata.normalize('NFKD', title)
    no_accents = ''.join([c for c in nfkd_form if not unicodedata.combining(c)])
    
    # 2. Replace special characters with underscores
    name = re.sub(r"[ /–—]+", "_", no_accents)          # replace spaces/dashes/slashes with underscore
    name = re.sub(r"[^A-Za-z0-9_!'’-]", "", name)      # remove any other symbol (keep ! for some cases)
    
    # 3. Collapse multiple underscores and trim
    name = re.sub(r"_+", "_", name).strip("_")
    
    return name


In [10]:
# function that replaces the directory name for specific journal names
def replace_dir_name(batch: pd.DataFrame, journal_name: str, dir_name: str) -> pd.DataFrame:
    batch.loc[
        batch['journal_name'] == journal_name,
        'directory_name'
    ] = dir_name
    return batch

## BCUL Media List

In [11]:
media_df = pd.read_csv('/rcp-scratch/iccluster040_scratch/students/banuls/impresso-text-acquisition/text_preparation/data/sample_data/BCUL/bcul_media_list.csv', dtype=str)

In [12]:
media_df.head()

,Unnamed: 0,Collection name,Journal Name,Start Year (in Scriptorium),End Year (in Scriptorium),Number of issues,"@TeamHistory, please express your interest 👇",Data Category for Data Sharing Agreement,Data Acquisition Batch,Data Transfer Batch (what has been transfered with hard disk),Extension of mit file,Impresso 1,Formula (if title in B was already in previous sheet),Question/Notes
0,1,24 Heures / FAL,24 Heures,1972,2012,12342,high,Category 3 - no agreement,Batch 2 – partly public,To add later,NaN,2,No,NaN
1,222,Revues UNIL,Allez savoir!,1994,2023,85,medium,Category 1.5 - Orphan,Batch 3 – restricted,Bath 4 (08/25),NaN,NaN,No,NaN
2,20,Journaux satiriques vaudois,Almanach Balthasar,1922,1926,4,medium,Category 1.5 - Orphan,Batch 3 – restricted,Batch 3 (12/24),NaN,3,Almanach Balthasar,NaN
3,21,Journaux satiriques vaudois,Almanach de Chalamala,1911,1919,5,medium,Category 1.5 - Orphan,Batch 3 – restricted,Batch 3 (12/24),NaN,4,No,Dans la liste précédente: Almanach de Chalaman...
4,5,Almanachs vaudois,Almanach de Lausanne,1674,1911,121,medium,Category 1.5 - Orphan,Batch 1.5,Not to include,NaN,NaN,No,NaN


In [13]:
# find journal named "L’ami_de_la_vérité__journal_du_Canton_de_Vaud"
temp = media_df[media_df['Journal Name'] == "L'ami de la vérité: journal du Canton de Vaud"]
temp

,Unnamed: 0,Collection name,Journal Name,Start Year (in Scriptorium),End Year (in Scriptorium),Number of issues,"@TeamHistory, please express your interest 👇",Data Category for Data Sharing Agreement,Data Acquisition Batch,Data Transfer Batch (what has been transfered with hard disk),Extension of mit file,Impresso 1,Formula (if title in B was already in previous sheet),Question/Notes
89,87,Nouvelliste vaudois,L'ami de la vérité: journal du Canton de Vaud,1822,1823,81,high,Category 1– public domain,Batch 1 – public domain,Batch 1 (21/12/23),XML (needs API),16,L'ami de la vérité: journal du Canton de Vaud,NaN


Cleaning the Dataframe, droping useless columns, keeping batches 3 and 4 only

In [14]:
media_df = media_df.drop('Data Acquisition Batch', axis=1)
media_df = media_df.rename(columns={"Data Transfer Batch (what has been transfered with hard disk)": "data_transfer_batch", "Collection name": "collection_name", "Journal Name": "journal_name", "Directory Name": "directory_name", "Start Year (in Scriptorium)": "start_year", "End Year (in Scriptorium)": "end_year", "Number of issues": "number_of_issues"})
media_df = media_df[(media_df['data_transfer_batch'] != 'To add later') & (media_df['data_transfer_batch'] != 'Not to include')]

media_df = media_df.drop([media_df.columns[0], 'Question/Notes', 'Extension of mit file', 'Data Category for Data Sharing Agreement', '@TeamHistory, please express your interest 👇', 'Impresso 1', 'Formula (if title in B was already in previous sheet)'], axis=1)
media_df.reset_index(drop=True, inplace=True)
media_df.head(5)

,collection_name,journal_name,start_year,end_year,number_of_issues,data_transfer_batch
0,Revues UNIL,Allez savoir!,1994,2023,85,Bath 4 (08/25)
1,Journaux satiriques vaudois,Almanach Balthasar,1922,1926,4,Batch 3 (12/24)
2,Journaux satiriques vaudois,Almanach de Chalamala,1911,1919,5,Batch 3 (12/24)
3,Annuaire vaudois,Almanach pour le commerce,1832,1832,1,Batch 1 (21/12/23)
4,Journaux satiriques vaudois,Arrêtez-les tous !,1972,1972,1,Batch 3 (12/24)


In [15]:

batch1_2 = media_df[(media_df['data_transfer_batch'] == 'Batch 1 (21/12/23)') | (media_df['data_transfer_batch'] == 'Batch 2 (26/02/24)')].reset_index(drop=True)


In [16]:
# find journal named "L’ami_de_la_vérité__journal_du_Canton_de_Vaud"
temp = batch1_2[batch1_2['journal_name'] == "La Veveysanne - La Patrie"]
temp

,collection_name,journal_name,start_year,end_year,number_of_issues,data_transfer_batch
28,Presse de l'Est vaudois,La Veveysanne - La Patrie,1840,1844,216,Batch 2 (26/02/24)


Modify 'La_Veveysanne - La Patrie' entry

In [17]:
# delete journal_name "La Veveysanne - La Patrie" and add "La Veveysanne" and "La Patrie"
batch1_2 = batch1_2[batch1_2['journal_name'] != "La Veveysanne - La Patrie"]
new_rows = pd.DataFrame({
    'collection_name': ["Presse de l'Est vaudois", "Presse de l'Est vaudois"],
    'journal_name': ['La Veveysanne', 'La Patrie'],
    'start_year': ['1840', '1841'],
    'end_year': ['1844', '1844'],
    'number_of_issues': [np.nan, np.nan],
    'data_transfer_batch': ['Batch 1 (21/12/23)', 'Batch 1 (21/12/23)'],
    'directory_name': ['La_Veveysanne', 'La_Patrie'],    
})
# append new rows to batch1_2 in alphabetical order
batch1_2 = pd.concat([batch1_2, new_rows], ignore_index=True)
batch1_2 = batch1_2.sort_values(by='journal_name').reset_index(drop=True)

In [18]:
batch1_2

,collection_name,journal_name,start_year,end_year,number_of_issues,data_transfer_batch,directory_name
0,Annuaire vaudois,Almanach pour le commerce,1832,1832,1,Batch 1 (21/12/23),NaN
1,Bulletins du Grand Conseil,Bulletins du Grand Conseil,1829,2020,619,Batch 2 (26/02/24),NaN
2,Journaux satiriques vaudois,Castigat ridendo mores,1879,1880,6,Batch 1 (21/12/23),NaN
3,Presse de l'Est vaudois,Courrier du Léman,1882,1886,445,Batch 1 (21/12/23),NaN
4,Journaux satiriques vaudois,Croquis vaudois,1884,1887,55,Batch 1 (21/12/23),NaN
5,Presse de la Côte vaudoise,Feuille d'avis de Morges 1,1854,1854,3,Batch 2 (26/02/24),NaN
6,"Presse de la Broye, du Jorat et du Gros-de-Vaud",Feuille d'avis de Moudon,1859,1859,2,Batch 1 (21/12/23),NaN
7,Presse de la Côte vaudoise,Feuille d'avis de Nyon,1849,1867,86,Batch 1 (21/12/23),NaN
8,"Presse de la Broye, du Jorat et du Gros-de-Vaud",Feuille d'avis de Payerne,1843,1845,2,Batch 1 (21/12/23),NaN
9,"Presse de la Broye, du Jorat et du Gros-de-Vaud","Feuille d'avis de Payerne, Moudon et Avenches",1852,1852,1,Batch 1 (21/12/23),NaN


In [19]:
# keep only batch 3 and 4
batch3_4 = media_df[(media_df['data_transfer_batch'] == 'Bath 4 (08/25)') | (media_df['data_transfer_batch'] == 'Batch 3 (12/24)')].reset_index(drop=True)


New column with the Journal names as they are in the File system /mnt/project_impresso/original/BCUL/

In [20]:
titles1_2 = list(batch1_2['journal_name'])
titles3_4 = list(batch3_4['journal_name'])

In [21]:
len(titles1_2)

50

In [22]:
dirnames = os.listdir(PATH_FILE_SYSTEM)

normalized_dirs = {d.lower(): d for d in dirnames}  # lowercase dict for robust matching

matches = {}
for title in titles1_2:
    norm = normalize_title(title).lower()
    # find closest match
    match = next((v for k, v in normalized_dirs.items() if norm in k), None)
    matches[title] = match or "NOT FOUND"

for k, v in matches.items():
    batch1_2.loc[batch1_2['journal_name'] == k, 'directory_name'] = v


In [23]:
dirnames = os.listdir(PATH_FILE_SYSTEM)

normalized_dirs = {d.lower(): d for d in dirnames}  # lowercase dict for robust matching

matches = {}
for title in titles3_4:
    norm = normalize_title(title).lower()
    # find closest match
    match = next((v for k, v in normalized_dirs.items() if norm in k), None)
    matches[title] = match or "NOT FOUND"

for k, v in matches.items():
    batch3_4.loc[batch3_4['journal_name'] == k, 'directory_name'] = v


In [24]:
not_found_names1_2 = list(batch1_2[batch1_2['directory_name'] == "NOT FOUND"]['journal_name'])
not_found_names1_2



['Courrier du Léman',
 "Feuille d'avis de Payerne, Moudon et Avenches",
 'Journal de Vevey et des étrangers',
 'Journal helvétique',
 "L'Observateur du Léman",
 "L'ami de la vérité: journal du Canton de Vaud",
 'La Guêpe [1]',
 'La Guêpe [2]',
 'Le Charivari [1]',
 'Le Charivari [2]',
 'Le Moustique (Vevey)',
 'Le Père Jérôme',
 'Le Touriste / La Suisse Illustrée',
 'Messager boîteux']

In [25]:
batch1_2 = replace_dir_name(batch1_2, 'Courrier du Léman', 'Courrier_du_Léman')
batch1_2 = replace_dir_name(batch1_2, "Feuille d'avis de Payerne, Moudon et Avenches", "Feuille_d'avis_de_Payerne,_Moudon_et_Avenches")
batch1_2 = replace_dir_name(batch1_2, 'Journal de Vevey et des étrangers', "Journal_de_vevey_et_des_étrangers")
batch1_2 = replace_dir_name(batch1_2, 'Journal helvétique', 'Journal_helvétique')
batch1_2 = replace_dir_name(batch1_2, "L'ami de la vérité: journal du Canton de Vaud", "L’ami_de_la_vérité__journal_du_Canton_de_Vaud")
batch1_2 = replace_dir_name(batch1_2, "L'Observateur du Léman", "L'Observateur_du_Léman")
batch1_2 = replace_dir_name(batch1_2, 'La Guêpe [1]', 'La_Guêpe_[1]')
batch1_2 = replace_dir_name(batch1_2, 'La Guêpe [2]', 'La_Guêpe_[2]')
batch1_2 = replace_dir_name(batch1_2, 'La Veveysanne - La Patrie', 'La_Veveysanne_–_La_Patrie')
batch1_2 = replace_dir_name(batch1_2, 'Le Charivari [1]', 'Le_Charivari_[1]')
batch1_2 = replace_dir_name(batch1_2, 'Le Charivari [2]', 'Le_Charivari_[2]')
batch1_2 = replace_dir_name(batch1_2, 'Le Moustique (Vevey)', 'Le_Moustique_(Vevey)')
batch1_2 = replace_dir_name(batch1_2, 'Le Père Jérôme', 'Le_Père_Jérôme')
batch1_2 = replace_dir_name(batch1_2, 'Le Touriste / La Suisse Illustrée', 'Le_Touriste__La_Suisse_illustrée')
batch1_2 = replace_dir_name(batch1_2, 'Messager boîteux', 'Messager_boîteux')
batch1_2 = replace_dir_name(batch1_2, 'La Fronde', 'La_Fronde')
batch1_2 = replace_dir_name(batch1_2, "Feuille d'avis de Payerne", "Feuille_d'avis_de_Payerne")


In [26]:
not_found_names3_4 = list(batch3_4[batch3_4['directory_name'] == "NOT FOUND"]['journal_name'])
not_found_names3_4



['Croire / Ensemble',
 'Echo du Petit-Lac/Rive droite Rive gauche',
 "Feuille d'avis de Morges 2 / L'Ami de Morges",
 'Gland Eco/Gland Cité',
 "Journal de Chateau -d'Oex",
 'Journal des Brandons de Payerne',
 'L’Éternité',
 'Le Bonjour de Jack Rollan (2)',
 'Le Moustique (Lausanne)',
 'Yverdon Revue / Flash Revue']

Replacing manually the names to match the file system directories

In [27]:
batch3_4 = replace_dir_name(batch3_4, "Croire / Ensemble", 'Croire__Ensemble')
batch3_4 = replace_dir_name(batch3_4, 'Echo du Petit-Lac/Rive droite Rive gauche', 'Echo_du_Petit-LacRive_droite_Rive_gauche')
batch3_4 = replace_dir_name(batch3_4, "Feuille d'avis de Morges 2 / L'Ami de Morges", "Feuille_d'avis_de_Morges_2__L'Ami_de_Morges")
batch3_4 = replace_dir_name(batch3_4, "Gland Eco/Gland Cité", 'Gland_EcoGland_Cite')
batch3_4 = replace_dir_name(batch3_4, "Journal de Chateau -d'Oex", "Journal_de_Chateau-d'Oex")
batch3_4 = replace_dir_name(batch3_4, "Journal des Brandons de Payerne", 'Journal_de_Payerne')
batch3_4 = replace_dir_name(batch3_4, "L’Éternité", "L'Eternite")
batch3_4 = replace_dir_name(batch3_4, 'Le Bonjour de Jack Rollan (2)', "Le_Bonjour_de_Jack_Rollan_(2)")
batch3_4 = replace_dir_name(batch3_4, 'Le Moustique (Lausanne)', "Le_Moustique_(Lausanne)")
batch3_4 = replace_dir_name(batch3_4, 'Yverdon Revue / Flash Revue', 'Yverdon_Revue__Flash_Revue')
batch3_4 = replace_dir_name(batch3_4, "Nouvelle revue", "Nouvelle_revue")
batch3_4 = replace_dir_name(batch3_4, "L'Echo", "L'Echo")


batch3_4 = replace_dir_name(batch3_4, "Journal des Brandons de Payerne", None)

### PAS DE JOURNAL des Brandons de Payerne

In [28]:
batch3_4.head(5)

,collection_name,journal_name,start_year,end_year,number_of_issues,data_transfer_batch,directory_name
0,Revues UNIL,Allez savoir!,1994,2023,85,Bath 4 (08/25),Allez_savoir!
1,Journaux satiriques vaudois,Almanach Balthasar,1922,1926,4,Batch 3 (12/24),Almanach_Balthasar
2,Journaux satiriques vaudois,Almanach de Chalamala,1911,1919,5,Batch 3 (12/24),Almanach_de_Chalamala
3,Journaux satiriques vaudois,Arrêtez-les tous !,1972,1972,1,Batch 3 (12/24),Arretez-les_tous_!
4,"Presse de la Broye, du Jorat et du Gros-de-Vaud",Aux quatre coins du Mont,1974,2022,282,Bath 4 (08/25),Aux_quatre_coins_du_MontMont_Journal


Check if directory names are the same

In [29]:
# check if directory names in PATH_FILE_SYSTEM are the same as in the dataframe
dirnames = os.listdir(PATH_FILE_SYSTEM)
set_dirnames = set(dirnames)
set_batch3_4_dirs = set(batch3_4['directory_name'])
# check if the names in set_batch_dirs are in set_dirnames
set_batch3_4_dirs - set_dirnames

{None}

In [30]:
set_batch1_2_dirs = set(batch1_2['directory_name'])
# check if the names in set_batch_dirs are in set_dirnames
set_batch1_2_dirs - set_dirnames

set()

In [31]:
(set_dirnames - set_batch1_2_dirs) - set_batch3_4_dirs

{'.DS_Store',
 'La_Veveysanne_–_La_Patrie',
 'Le_Phare_du_Léman',
 'OLD',
 'wrong_BCUL'}

#### Duplicated journals 

In [32]:
len(set_batch1_2_dirs)

49

In [33]:
len(batch1_2['directory_name'])

50

In [34]:
len(batch3_4['directory_name'])

128

In [35]:
# count how many directories in each sets
len(set_dirnames), len(set_batch3_4_dirs)

(181, 128)

In [36]:
# change directory name 'La_Veveysanne__La_Patrie' for 'La_Veveysanne_–_La_Patrie'
# batch1_2[batch1_2['journal_name']=='La Patrie']['directory_name'] = 'La_Veveysanne_–_La_Patrie/La_Patrie'
# batch1_2[batch1_2['journal_name']=='La Veveysanne']['directory_name'] = 'La_Veveysanne_–_La_Patrie/La_Veveysanne'

# do the same but with .loc
batch1_2.loc[
    batch1_2['journal_name']=='La Patrie',
    'directory_name'
] = 'La_Veveysanne_–_La_Patrie/La_Patrie'
batch1_2.loc[
    batch1_2['journal_name']=='La Veveysanne',
    'directory_name'
] = 'La_Veveysanne_–_La_Patrie/La_Veveysanne'


In [37]:
batch1_2[batch1_2['directory_name'].duplicated()]

,collection_name,journal_name,start_year,end_year,number_of_issues,data_transfer_batch,directory_name


In [38]:
# check which directories in the dataframe are duplicated
batch3_4[batch3_4['directory_name'].duplicated()]

,collection_name,journal_name,start_year,end_year,number_of_issues,data_transfer_batch,directory_name


All journal names from batch 1_2 and 3_4 and their corresponding directory names

In [39]:
# list of all journal names from batch 1_2 and 3_4 and their corresponding directory names, keeping their batch

all_batches = pd.concat([batch1_2, batch3_4], ignore_index=True)
all_batches = all_batches[['journal_name', 'directory_name', 'data_transfer_batch']]
all_batches = all_batches.sort_values(by='journal_name').reset_index(drop=True)
all_batches

,journal_name,directory_name,data_transfer_batch
0,Allez savoir!,Allez_savoir!,Bath 4 (08/25)
1,Almanach Balthasar,Almanach_Balthasar,Batch 3 (12/24)
2,Almanach de Chalamala,Almanach_de_Chalamala,Batch 3 (12/24)
3,Almanach pour le commerce,Almanach_pour_le_commerce,Batch 1 (21/12/23)
4,Arrêtez-les tous !,Arretez-les_tous_!,Batch 3 (12/24)
...,...,...,...
173,Revue photo-rigolotipique,Revue_photo-rigolotipique,Batch 3 (12/24)
174,Répertoires de Bulletins du Grand Conseil,Repertoires_de_Bulletins_du_Grand_Conseil,Batch 2 (26/02/24)
175,Terre vuilleraine,Terre_vuilleraine,Bath 4 (08/25)
176,Travaux des Constituantes,Travaux_des_Constituantes,Batch 2 (26/02/24)


## Fetching and Finding Aliases for the Titles


Now, we have all titles from batch 1_2 and batch 3_4 with their journal names and their directory names, as in the file system. We can now fetch the aliases from batch 1_2 and find some for batch 3_4.

In [57]:
aliases1_2 = pd.read_json('/rcp-scratch/iccluster040_scratch/students/banuls/impresso-text-acquisition/text_preparation/data/sample_data/BCUL/bcul_aliases.json').T

I don't understand the 'bin_title' col. Why do you transforme it this way ? in the file system it's not written like this. Do you remember why you did this ?

In [58]:
list_aliases1_2 = list(aliases1_2.alias)

Aliases generated by Chat-GPT, asked on the same format than the aliases of batch 1_2.

In [59]:
dic_aliases3_4 = {
    "Allez_savoir!": "AS", #AS!
    "Almanach_Balthasar": "ABal",
    "Almanach_de_Chalamala": "AChal",
    "Arretez-les_tous_!": "ALT",
    "Aux_quatre_coins_du_MontMont_Journal": "AQCMJ",
    "Bonne_nouvelle": "BN",
    "Chut": "CHU",
    "Clin_d'oeil": "CLOEIL", #"CDO",
    "Confiance": "CONF",
    "Courrier_de_la_Cote": "CC", #"CCot",
    "Courrier_de_la_Montagne": "CM", #"CMon",
    "Courrier_de_Leysin": "CLey",
    "Courrier_de_Vevey": "COVE", #"CV",
    "Croire__Ensemble": "croire", #"CE",
    "Domaine_Public": "DP", #"DPub",
    "Echo_de_la_Montagne": "ECM", #"EDM",
    "Echo_du_Petit-LacRive_droite_Rive_gauche": "RDRG", #"EPL",
    "Feuille_d'avis_d'Aigle": "FAA",
    "Feuille_d'avis_d'Avenches": "FAAV", #"FAVe",
    "Feuille_d'avis_d'Orbe": "FAOR", #"FAO",
    "Feuille_d'avis_de_Biere_1": "FAB1",
    "Feuille_d'avis_de_Biere_2": "FAB2",
    "Feuille_d'avis_de_Grandson": "FAG",
    "Feuille_d'avis_de_la_Cote": "FAC",
    "Feuille_d'avis_de_Montreux": "FAMX",
    "Feuille_d'avis_de_Morges_2__L'Ami_de_Morges": "FAM2",
    "Feuille_d'avis_de_Sainte-Croix": "FASC",
    "Feuille_d'avis_de_Vallorbe": "FAVA", #"FAVa",
    "Feuille_d'avis_des_cercles_de_Lucens_et_Granges": "FACLG",
    "Gland_-_Bulletin_d'information_municipal": "BIMG", #"GBI",
    "Gland_EcoGland_Cite": "GLEC", #"GEGC",
    "Gribouille_et_Redzipet": "GER",
    "Journal_-_Feuille_d'avis_de_Coppet": "JCOP", #"JFAC",
    "Journal_d'Aubonne": "JA", #"JAub",
    "Journal_de_Bex": "JBEX", #"JBex",
    "Journal_de_Bussigny": "JB", #"JBus",
    "Journal_de_Chateau-d'Oex": "JCO", #"JCdO",
    "Journal_de_Cossonay": "JC", #"JCos",
    "Journal_de_l'Isle": "JLI", #"JLIs",
    "Journal_de_la_Sarraz": "JLS", #"JSar",
    "Journal_de_Leysin": "JL", #"JLey",
    "Journal_de_Montreux": "JOM", #"JMx",
    "Journal_de_Morges": "JM", #"JMorg",
    "Journal_de_Moudon": "JMOU", #"JMou",
    "Journal_de_Nyon": "JN", #"JNy",
    "Journal_de_Payerne": "JPA", #"JPay",
    "Journal_de_Prilly": "JP", #"JPri",
    "Journal_de_Pully": "JPU", #"JPul",
    "Journal_de_Rolle": "JR", #"JRol",
    "Journal_de_Vallorbe": "JoVA", #"JVal",
    "Journal_des_Brandons_de_Moudon": "JBM",
    None: None,
    "Journal_du_Chablais": "JDC", #"JChab",
    "Journal_du_district_d'Avenches": "JAVE", #"JDA",
    "Journal_du_Pays_d'Enhaut": "JPE",
    "Journal_et_liste_des_etrangers_de_Montreux": "JLEM",
    "Journal_officiel_de_Vevey": "JOV",
    "L'Arbalete": "ARB",
    "L'Avenir_de_Vevey": "AVVE", #"AVV",
    "L'Echo": "ECHO",
    "L'Echo_de_la_Broye": "ECHOB", #"EB1",
    "L'Echo_du_Gros-de-Vaud": "ECHOGV", #"EGV",
    "L'Echo_du_Rhone": "ECR", #"ERh",
    "L'Echo_rollois": "ER", #"ERo",
    "L'Eternite": "ETER",
    "L'Etincelle": "ETIN",
    "L'Eveil": "EVE", #"EVEI",
    "L'Omnibus": "OMN",
    "La_Broye_Broye_Hebdo": "BROYE", #"LBH",
    "La_Cote": "LC", #"LCot",
    "La_Crecelle": "CREC",
    "La_detonation": "DETO",
    "La_Fronde_de_Lausanne": "FDL",
    "La_Marge": "MARG",
    "La_Mouette": "MOU",
    "La_Nation": "NAT",
    "La_Petite_revue": "RLP", #"LPR",
    "La_Pomme": "POM",
    "La_Revue_de_Lausanne": "RL", #"RDL",
    "La_Revue_dimanche": "RLD", #"RDIM",
    "La_Tribune_de_Montreux": "TDM",
    "La_Tribune_de_Vevey": "TRVE", #"TDV",
    "La_Vague": "VAG",
    "La_voix_du_peuple_vaudois": "VPV",
    "Lausanne-Soir": "RLS", #"LS",
    "Le_Bistouri": "BIST",
    "Le_Bon_jour_de_Jack_Rollan": "Rollan1", #"BJJR1",
    "Le_Bonjour_de_Jack_Rollan_(2)": "Rollan2", #"BJJR2",
    "Le_Canard_libre": "CLib",
    "Le_Canard_rit": "CRit",
    "Le_Clairon_du_Nord": "CLN",
    "Le_Cotterg": "COTT",
    "Le_Courrier_de_la_Broye_et_du_Jorat": "CBJ",
    "Le_Courrier_de_Prilly": "COPU", #"CPri",
    "Le_Democrate_de_Payerne": "DPA", #"DDP",
    "Le_Frondeur": "FRON",
    "Le_Jorat": "JOR",
    "Le_Journal_de_Jack_Rollan": "Rollan3", #"JJR",
    "Le_Jura_vaudois": "JURV",
    "Le_Kangourou": "KANG",
    "Le_Magazine": "MA", #"MAG",
    "Le_Moustique_(Lausanne)": "MOU2",
    "Le_Passe-Temps": "PT",
    "Le_Petit_Paysan_-_Region_Evasion": "PEP", #"PPE",
    "Le_poisson_dechaine": "PDech",
    "Le_Progres_de_Chateau_d'Oex": "PCO",
    "Le_Progres_de_Morges": "PMorg",
    "Le_Semeur": "SEM",
    "Le_Veveysan_2": "VVS2",
    "Les_Alpes_vaudoises": "ALP",
    "Liste_des_etrangers_de_Bex": "LEB",
    "Mepris": "MEP",
    "Messager_de_Montreux": "MEM", #"MMX",
    "Morges_actualite_-_Morges_Hebdo": "MH",
    "Nouvelle_revue": "NRev",
    "Nouvelle_revue_de_Lausanne_et_du_Pays_de_Vaud": "NRL",
    "Nouvelle_revue_hebdo": "NRH",
    "Ouest_Informations": "OUI", #"OI",
    "Pages_illustrees": "PI",
    "Pour_la_famille": "PLF",
    "Presse_Lausannoise": "PL",
    "Prilly_Nouveau": "PN", #"PNouv",
    "Propos_vaudois": "PV",
    "Reflets_de_la_Cote": "REC", #"RDC",
    "Revue_morgienne": "RMorg",
    "Revue_photo-rigolotipique": "RPR",
    "Terre_vuilleraine": "TV",
    "Yverdon_Revue__Flash_Revue": "YRE", #"YRFR"
}



In [60]:
# check if the aliases of batch 3_4 are different from the ones in batch 1_2
len(dic_aliases3_4.values()), len(set(dic_aliases3_4.values()) - set(list_aliases1_2))

(128, 128)

In [61]:
# find the aliases in dic_aliases3_4 that are the same as in list_aliases1_2
set(dic_aliases3_4.values()) & set(list_aliases1_2)

set()

In [62]:
len(ALL_MEDIA)

514

In [63]:
# check if the aliases of batch 3_4 are diffenrent than the aliases of ALL_MEDIA
len(set(dic_aliases3_4.values()) - set(ALL_MEDIA))

128

Perfect !

## Adding new aliases

In [103]:
batch3_4_with_aliases = batch3_4.copy()

In [104]:
batch3_4_with_aliases['alias'] = batch3_4_with_aliases['directory_name'].map(dic_aliases3_4)

In [105]:
batch3_4_with_aliases

,collection_name,journal_name,start_year,end_year,number_of_issues,data_transfer_batch,directory_name,alias
0,Revues UNIL,Allez savoir!,1994,2023,85,Bath 4 (08/25),Allez_savoir!,AS
1,Journaux satiriques vaudois,Almanach Balthasar,1922,1926,4,Batch 3 (12/24),Almanach_Balthasar,ABal
2,Journaux satiriques vaudois,Almanach de Chalamala,1911,1919,5,Batch 3 (12/24),Almanach_de_Chalamala,AChal
3,Journaux satiriques vaudois,Arrêtez-les tous !,1972,1972,1,Batch 3 (12/24),Arretez-les_tous_!,ALT
4,"Presse de la Broye, du Jorat et du Gros-de-Vaud",Aux quatre coins du Mont,1974,2022,282,Bath 4 (08/25),Aux_quatre_coins_du_MontMont_Journal,AQCMJ
...,...,...,...,...,...,...,...,...
123,Presse de la Côte vaudoise,Reflets de la Côte,1980,1982,26,Batch 3 (12/24),Reflets_de_la_Cote,REC
124,Presse de la Côte vaudoise,Revue morgienne,1902,1905,33,Batch 3 (12/24),Revue_morgienne,RMorg
125,Journaux satiriques vaudois,Revue photo-rigolotipique,1902,1902,1,Batch 3 (12/24),Revue_photo-rigolotipique,RPR
126,"Presse de la Broye, du Jorat et du Gros-de-Vaud",Terre vuilleraine,1953,2010,2579,Bath 4 (08/25),Terre_vuilleraine,TV


## Finding format of the Mit file

In [128]:
def find_mit_extension(directory_name: str) -> str: 
    extensions = set()
    journal_dir = os.path.join(PATH_FILE_SYSTEM, directory_name)
    for dir_path, dirs, files in os.walk(journal_dir):
        if len(files)>1 and 'solr' not in dir_path:
            # print(dir_path)
            for file in files:
                if 'mit' in file:
                    # print(file.split('.')[-1])
                    extensions.add(file.split('.')[-1])

    if len(extensions) == 1:
        return extensions.pop()
    else:
        return None

### DO NOT RUN AGAIN, TAKES 4 HOURS

In [42]:
batch3_4.head(2)

,collection_name,journal_name,start_year,end_year,number_of_issues,data_transfer_batch,directory_name
0,Revues UNIL,Allez savoir!,1994,2023,85,Bath 4 (08/25),Allez_savoir!
1,Journaux satiriques vaudois,Almanach Balthasar,1922,1926,4,Batch 3 (12/24),Almanach_Balthasar


In [ ]:
batch3_4_aliases_mit = batch3_4_with_aliases.copy()
batch3_4_aliases_mit['mit_file_type'] = batch3_4_with_aliases['directory_name'].apply(find_mit_extension)


/mnt/project_impresso/original/BCUL/Allez_savoir!/2016/05/00/200271
json
/mnt/project_impresso/original/BCUL/Allez_savoir!/2016/01/00/200274
json
/mnt/project_impresso/original/BCUL/Allez_savoir!/2016/09/00/200867
json
/mnt/project_impresso/original/BCUL/Allez_savoir!/1995/04/01/125746
xml
/mnt/project_impresso/original/BCUL/Allez_savoir!/1995/06/01/125758
xml
/mnt/project_impresso/original/BCUL/Allez_savoir!/1995/10/01/125769
xml
/mnt/project_impresso/original/BCUL/Allez_savoir!/2020/01/00/521893
json
/mnt/project_impresso/original/BCUL/Allez_savoir!/2020/09/00/521892
json
/mnt/project_impresso/original/BCUL/Allez_savoir!/2020/12/00/605914
json
/mnt/project_impresso/original/BCUL/Allez_savoir!/2013/05/01/125796
xml
/mnt/project_impresso/original/BCUL/Allez_savoir!/2013/01/01/125795
xml
/mnt/project_impresso/original/BCUL/Allez_savoir!/2013/09/00/196979
json
/mnt/project_impresso/original/BCUL/Allez_savoir!/2014/05/00/196980
json
/mnt/project_impresso/original/BCUL/Allez_savoir!/2014/0

In [106]:
# create dic journal name as key and mit extension as value from the json bcul3_4.json
with open('/rcp-scratch/iccluster040_scratch/students/banuls/impresso-text-acquisition/text_preparation/data/sample_data/BCUL/bcul3_4.json', 'r') as f:
    bcul3_4_data = json.load(f)
dic_journal_mit = {entry['Directory Name']: entry['mit_extension'] for entry in bcul3_4_data}

In [ ]:
# create dic journal name as key and mit extension as value
dic_journal_mit = dict(zip(batch3_4_aliases_mit['journal_name'], batch3_4_aliases_mit['mit_file_type']))

NameError: name 'batch3_4_aliases_mit' is not defined

In [107]:
# add col local path, for example /mnt/project_impresso/original/BCUL/Allez_savoir!
batch3_4_with_aliases['local_path'] = PATH_FILE_SYSTEM + batch3_4_with_aliases['directory_name']
batch3_4_with_aliases

,collection_name,journal_name,start_year,end_year,number_of_issues,data_transfer_batch,directory_name,alias,local_path
0,Revues UNIL,Allez savoir!,1994,2023,85,Bath 4 (08/25),Allez_savoir!,AS,/mnt/project_impresso/original/BCUL/Allez_savoir!
1,Journaux satiriques vaudois,Almanach Balthasar,1922,1926,4,Batch 3 (12/24),Almanach_Balthasar,ABal,/mnt/project_impresso/original/BCUL/Almanach_B...
2,Journaux satiriques vaudois,Almanach de Chalamala,1911,1919,5,Batch 3 (12/24),Almanach_de_Chalamala,AChal,/mnt/project_impresso/original/BCUL/Almanach_d...
3,Journaux satiriques vaudois,Arrêtez-les tous !,1972,1972,1,Batch 3 (12/24),Arretez-les_tous_!,ALT,/mnt/project_impresso/original/BCUL/Arretez-le...
4,"Presse de la Broye, du Jorat et du Gros-de-Vaud",Aux quatre coins du Mont,1974,2022,282,Bath 4 (08/25),Aux_quatre_coins_du_MontMont_Journal,AQCMJ,/mnt/project_impresso/original/BCUL/Aux_quatre...
...,...,...,...,...,...,...,...,...,...
123,Presse de la Côte vaudoise,Reflets de la Côte,1980,1982,26,Batch 3 (12/24),Reflets_de_la_Cote,REC,/mnt/project_impresso/original/BCUL/Reflets_de...
124,Presse de la Côte vaudoise,Revue morgienne,1902,1905,33,Batch 3 (12/24),Revue_morgienne,RMorg,/mnt/project_impresso/original/BCUL/Revue_morg...
125,Journaux satiriques vaudois,Revue photo-rigolotipique,1902,1902,1,Batch 3 (12/24),Revue_photo-rigolotipique,RPR,/mnt/project_impresso/original/BCUL/Revue_phot...
126,"Presse de la Broye, du Jorat et du Gros-de-Vaud",Terre vuilleraine,1953,2010,2579,Bath 4 (08/25),Terre_vuilleraine,TV,/mnt/project_impresso/original/BCUL/Terre_vuil...


In [108]:
dic_journal_mit

{'Allez_savoir!': None,
 'Almanach_Balthasar': 'xml',
 'Almanach_de_Chalamala': 'xml',
 'Arretez-les_tous_!': 'xml',
 'Aux_quatre_coins_du_MontMont_Journal': 'json',
 'Bonne_nouvelle': 'json',
 'Chut': 'xml',
 "Clin_d'oeil": 'json',
 'Confiance': 'json',
 'Courrier_de_la_Cote': 'json',
 'Courrier_de_la_Montagne': 'json',
 'Courrier_de_Leysin': 'json',
 'Courrier_de_Vevey': 'json',
 'Croire__Ensemble': 'json',
 'Domaine_Public': None,
 'Echo_de_la_Montagne': 'json',
 'Echo_du_Petit-LacRive_droite_Rive_gauche': 'json',
 "Feuille_d'avis_d'Aigle": 'json',
 "Feuille_d'avis_d'Avenches": 'json',
 "Feuille_d'avis_d'Orbe": 'json',
 "Feuille_d'avis_de_Biere_1": 'json',
 "Feuille_d'avis_de_Biere_2": 'json',
 "Feuille_d'avis_de_Grandson": 'json',
 "Feuille_d'avis_de_la_Cote": 'json',
 "Feuille_d'avis_de_Montreux": 'json',
 "Feuille_d'avis_de_Morges_2__L'Ami_de_Morges": 'json',
 "Feuille_d'avis_de_Sainte-Croix": 'json',
 "Feuille_d'avis_de_Vallorbe": 'json',
 "Feuille_d'avis_des_cercles_de_Lucens_e

In [109]:
batch3_4_final = batch3_4_with_aliases.copy()
# add mit extension thanks to the dic_journal_mit
batch3_4_final['mit_file_type'] = batch3_4_final['directory_name'].map(dic_journal_mit)


In [110]:
batch3_4_final['mit_file_type']

0      None
1       xml
2       xml
3       xml
4      json
       ... 
123    json
124    json
125     xml
126    json
127    json
Name: mit_file_type, Length: 128, dtype: object

In [111]:
batch3_4_final = batch3_4_final[batch3_4['journal_name'] != 'Journal des Brandons de Payerne']


In [114]:

batch3_4_final[batch3_4_final['mit_file_type'].isnull()]

,collection_name,journal_name,start_year,end_year,number_of_issues,data_transfer_batch,directory_name,alias,local_path,mit_file_type


In [113]:
batch3_4_final['mit_file_type'] = batch3_4_final['mit_file_type'].fillna('both')


In [115]:
# build the dictionary
data_dict = {
    row["directory_name"]: row.to_dict()
    for _, row in batch3_4_final.iterrows()
}

# convert to JSON (pretty-printed)
json_output = json.dumps(data_dict, indent=4, ensure_ascii=False)

# optional: save to file
with open("/rcp-scratch/iccluster040_scratch/students/banuls/impresso-text-acquisition/text_preparation/data/sample_data/BCUL/bcul_aliases3_4.json", "w", encoding="utf-8") as f:
    f.write(json_output)

print(json_output)

{
    "Allez_savoir!": {
        "collection_name": "Revues UNIL",
        "journal_name": "Allez savoir!",
        "start_year": "1994",
        "end_year": "2023",
        "number_of_issues": "85",
        "data_transfer_batch": "Bath 4 (08/25)",
        "directory_name": "Allez_savoir!",
        "alias": "AS",
        "local_path": "/mnt/project_impresso/original/BCUL/Allez_savoir!",
        "mit_file_type": "both"
    },
    "Almanach_Balthasar": {
        "collection_name": "Journaux satiriques vaudois",
        "journal_name": "Almanach Balthasar",
        "start_year": "1922",
        "end_year": "1926",
        "number_of_issues": "4",
        "data_transfer_batch": "Batch 3 (12/24)",
        "directory_name": "Almanach_Balthasar",
        "alias": "ABal",
        "local_path": "/mnt/project_impresso/original/BCUL/Almanach_Balthasar",
        "mit_file_type": "xml"
    },
    "Almanach_de_Chalamala": {
        "collection_name": "Journaux satiriques vaudois",
        "journal_n

In [116]:
batch3_4_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 127 entries, 0 to 127
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   collection_name      127 non-null    object
 1   journal_name         127 non-null    object
 2   start_year           127 non-null    object
 3   end_year             127 non-null    object
 4   number_of_issues     127 non-null    object
 5   data_transfer_batch  127 non-null    object
 6   directory_name       127 non-null    object
 7   alias                127 non-null    object
 8   local_path           127 non-null    object
 9   mit_file_type        127 non-null    object
dtypes: object(10)
memory usage: 10.9+ KB


In [290]:
bcul3_4_file = '../text_preparation/data/sample_data/BCUL/bcul3_4.json'

batch3_4_final.to_json(bcul3_4_file, orient='records', force_ascii=False, indent=4)

In [55]:
json1_2 = pd.read_json('/rcp-scratch/iccluster040_scratch/students/banuls/impresso-text-acquisition/text_preparation/data/sample_data/BCUL/bcul_aliases.json').T

In [56]:
json3_4 = pd.read_json('/rcp-scratch/iccluster040_scratch/students/banuls/impresso-text-acquisition/text_preparation/data/sample_data/BCUL/bcul3_4.json')

In [57]:
# keep only alias, file type and bin_title
json1_2 = json1_2[['alias', 'file_type']]


In [58]:
json1_2 = json1_2.reset_index().rename(columns={"index": "journal_name"})

In [46]:
json3_4.head(2)

,Collection name,Journal Name,Start Year (in Scriptorium),End Year (in Scriptorium),Number of issues,Data Transfer Batch (with hard disk),Directory Name,alias,local_path,mit_extension
0,Revues UNIL,Allez savoir!,1994,2023,85,Bath 4 (08/25),Allez_savoir!,AS,/mnt/project_impresso/original/BCUL/Allez_savoir!,None
1,Journaux satiriques vaudois,Almanach Balthasar,1922,1926,4,Batch 3 (12/24),Almanach_Balthasar,ABal,/mnt/project_impresso/original/BCUL/Almanach_B...,xml


In [60]:
json3_4 = json3_4[['Directory Name', 'alias', 'mit_extension']]

In [61]:
json3_4 = json3_4.rename(columns={"mit_extension": "file_type", "Directory Name": "journal_name"})
json3_4

,journal_name,alias,file_type
0,Allez_savoir!,AS,None
1,Almanach_Balthasar,ABal,xml
2,Almanach_de_Chalamala,AChal,xml
3,Arretez-les_tous_!,ALT,xml
4,Aux_quatre_coins_du_MontMont_Journal,AQCMJ,json
...,...,...,...
123,Reflets_de_la_Cote,REC,json
124,Revue_morgienne,RMorg,json
125,Revue_photo-rigolotipique,RPR,xml
126,Terre_vuilleraine,TV,json


In [62]:
# Merge / concatenate both
json123_4 = pd.concat([json1_2, json3_4], ignore_index=True)

In [66]:
# save as json file with indent 4 and force_ascii False
bcul123_4_file = '/rcp-scratch/iccluster040_scratch/students/banuls/impresso-text-acquisition/text_preparation/data/sample_data/BCUL/bcul_all_aliases.json'
json123_4.to_json(bcul123_4_file, orient='records', force_ascii=False, indent=4)

-----------------------------------------------------------------------------------------------------------------------------------

### Create big json file with all information on batch 1, 2, 3 and 4

In [129]:
# load bcul_aliases3_4.json
bcul3_4_aliases = pd.read_json('/rcp-scratch/iccluster040_scratch/students/banuls/impresso-text-acquisition/text_preparation/data/sample_data/BCUL/bcul_aliases3_4.json').T
bcul3_4_aliases.head(2)

,collection_name,journal_name,start_year,end_year,number_of_issues,data_transfer_batch,directory_name,alias,local_path,mit_file_type
Allez_savoir!,Revues UNIL,Allez savoir!,1994,2023,85,Bath 4 (08/25),Allez_savoir!,AS,/mnt/project_impresso/original/BCUL/Allez_savoir!,both
Almanach_Balthasar,Journaux satiriques vaudois,Almanach Balthasar,1922,1926,4,Batch 3 (12/24),Almanach_Balthasar,ABal,/mnt/project_impresso/original/BCUL/Almanach_B...,xml


In [130]:
# load bcul_alises.json
bcul1_2_aliases = pd.read_json('/rcp-scratch/iccluster040_scratch/students/banuls/impresso-text-acquisition/text_preparation/data/sample_data/BCUL/bcul_aliases1_2.json').T
bcul1_2_aliases.head(2)


,alias,file_type,access_right,bin_title
Mercure_suisse,ME,xml,open_public,Mercure_suisse
Nouvelliste_suisse,NS,xml,open_public,Nouvelliste_suisse


In [131]:
batch1_2_final = batch1_2.copy()

In [132]:
# change where 'directory_name' == 'Le_Phare_du_Leman' to 'Le_Phare_du_Léman'
batch1_2_final.loc[
    batch1_2_final['directory_name'] == 'Le_Phare_du_Leman',
    'directory_name'
] = 'Le_Phare_du_Léman'

In [133]:
# replace 'directory_name' with 'bin_title' if it exists from bcul1_2_aliases 
for index, row in bcul1_2_aliases.iterrows():
    print(index, row['bin_title'])
    # check if row['bin_title'] is not null
    if pd.notnull(row['bin_title']):
        #print("Replacing:", index, "with", row['bin_title'])
        batch1_2_final.loc[
            batch1_2_final['directory_name'] == index,
            'directory_name_2'
        ] = row['bin_title']
    else:
        batch1_2_final.loc[
            batch1_2_final['directory_name'] == index,
            'directory_name_2'
        ] = index
    # add 'bin_title' column
    # batch1_2_final.loc[
    #     batch1_2_final['directory_name'] == index,
    #     'bin_title'
    # ] = row['bin_title']

Mercure_suisse Mercure_suisse
Nouvelliste_suisse Nouvelliste_suisse
Journal_helvétique Journal_helvÃ©tique
Messager_boîteux Messager_boÃ®teux
L’ami_de_la_vérité__journal_du_Canton_de_Vaud Lâami_de_la_vÃ©ritÃ©__journal_du_Canton_de_Vaud
Almanach_pour_le_commerce Almanach_pour_le_commerce
Le_Père_Jérôme Le_PÃ¨re_JÃ©rÃ´me
Le_Charivari_[1] Le_Charivari_[1]
Le_Phare_de_Nyon Le_Phare_de_Nyon
Le_Phare_du_Léman Le_Phare_du_LÃ©man
Journal_d'Yverdon_2 Journal_d'Yverdon_2
Feuille_d'avis_de_Payerne Feuille_d'avis_de_Payerne
Le_Grelot Le_Grelot
Le_peuple_souverain Le_peuple_souverain
Feuille_d'avis_de_Payerne,_Moudon_et_Avenches Feuille_d'avis_de_Payerne,_Moudon_et_Avenches
La_Guêpe_[1] La_GuÃªpe_[1]
Feuille_d'avis_de_Moudon Feuille_d'avis_de_Moudon
L'Observateur_du_Léman L'Observateur_du_LÃ©man
La_Griffe La_Griffe
Feuille_d'avis_de_Nyon Feuille_d'avis_de_Nyon
Gazette_des_Villages Gazette_des_Villages
La_Fronde La_Fronde
Le_Charivari_[2] Le_Charivari_[2]
Castigat_ridendo_mores Castigat_ridendo_mo

In [134]:
# show where directory_name_2 is null
batch1_2_final[batch1_2_final['directory_name_2'].isnull()]

,collection_name,journal_name,start_year,end_year,number_of_issues,data_transfer_batch,directory_name,directory_name_2
27,Presse de l'Est vaudois,La Patrie,1841,1844,NaN,Batch 1 (21/12/23),La_Veveysanne_–_La_Patrie/La_Patrie,NaN
29,Presse de l'Est vaudois,La Veveysanne,1840,1844,NaN,Batch 1 (21/12/23),La_Veveysanne_–_La_Patrie/La_Veveysanne,NaN


In [135]:
batch1_2_final.loc[
            batch1_2_final['journal_name'] == 'La Patrie',
            'directory_name_2'
        ] = 'La_Veveysanne_–_La_Patrie/La_Patrie'
batch1_2_final.loc[
            batch1_2_final['journal_name'] == 'La Veveysanne',
            'directory_name_2'
        ] = 'La_Veveysanne_–_La_Patrie/La_Veveysanne'

In [136]:
batch1_2_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   collection_name      50 non-null     object
 1   journal_name         50 non-null     object
 2   start_year           50 non-null     object
 3   end_year             50 non-null     object
 4   number_of_issues     48 non-null     object
 5   data_transfer_batch  50 non-null     object
 6   directory_name       50 non-null     object
 7   directory_name_2     50 non-null     object
dtypes: object(8)
memory usage: 3.3+ KB


In [137]:
# create dic with 'bin_title' if it exists else index as key and 'alias' as value
dic_aliases1_2 = {}
for index, row in bcul1_2_aliases.iterrows():
    #key = row['bin_title'] if pd.notnull(row['bin_title']) else index
    key = index
    dic_aliases1_2[key] = row['alias']
    

In [138]:
# use dic_aliases1_2 to add alias column to batch1_2_final
batch1_2_final['alias'] = batch1_2_final['directory_name'].map(dic_aliases1_2)
batch1_2_final.head(5)

,collection_name,journal_name,start_year,end_year,number_of_issues,data_transfer_batch,directory_name,directory_name_2,alias
0,Annuaire vaudois,Almanach pour le commerce,1832,1832,1,Batch 1 (21/12/23),Almanach_pour_le_commerce,Almanach_pour_le_commerce,ACI
1,Bulletins du Grand Conseil,Bulletins du Grand Conseil,1829,2020,619,Batch 2 (26/02/24),Bulletins_du_Grand_Conseil,Bulletins_du_Grand_Conseil,RN
2,Journaux satiriques vaudois,Castigat ridendo mores,1879,1880,6,Batch 1 (21/12/23),Castigat_ridendo_mores,Castigat_ridendo_mores,Castigat
3,Presse de l'Est vaudois,Courrier du Léman,1882,1886,445,Batch 1 (21/12/23),Courrier_du_Léman,Courrier_du_LÃ©man,CL
4,Journaux satiriques vaudois,Croquis vaudois,1884,1887,55,Batch 1 (21/12/23),Croquis_vaudois,Croquis_vaudois,Croquis


In [139]:
# add 'alias' for 'La Patrie' and 'La Veveysanne' 
batch1_2_final.loc[
    batch1_2_final['journal_name'] == 'La Patrie',
    'alias'
] = 'PAT'
batch1_2_final.loc[
    batch1_2_final['journal_name'] == 'La Veveysanne',
    'alias'
] = 'VVS'

In [140]:
batch1_2_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   collection_name      50 non-null     object
 1   journal_name         50 non-null     object
 2   start_year           50 non-null     object
 3   end_year             50 non-null     object
 4   number_of_issues     48 non-null     object
 5   data_transfer_batch  50 non-null     object
 6   directory_name       50 non-null     object
 7   directory_name_2     50 non-null     object
 8   alias                50 non-null     object
dtypes: object(9)
memory usage: 3.6+ KB


In [141]:
batch1_2_final.head(5)

,collection_name,journal_name,start_year,end_year,number_of_issues,data_transfer_batch,directory_name,directory_name_2,alias
0,Annuaire vaudois,Almanach pour le commerce,1832,1832,1,Batch 1 (21/12/23),Almanach_pour_le_commerce,Almanach_pour_le_commerce,ACI
1,Bulletins du Grand Conseil,Bulletins du Grand Conseil,1829,2020,619,Batch 2 (26/02/24),Bulletins_du_Grand_Conseil,Bulletins_du_Grand_Conseil,RN
2,Journaux satiriques vaudois,Castigat ridendo mores,1879,1880,6,Batch 1 (21/12/23),Castigat_ridendo_mores,Castigat_ridendo_mores,Castigat
3,Presse de l'Est vaudois,Courrier du Léman,1882,1886,445,Batch 1 (21/12/23),Courrier_du_Léman,Courrier_du_LÃ©man,CL
4,Journaux satiriques vaudois,Croquis vaudois,1884,1887,55,Batch 1 (21/12/23),Croquis_vaudois,Croquis_vaudois,Croquis


In [142]:
# add local path
batch1_2_final['local_path'] = PATH_FILE_SYSTEM + batch1_2_final['directory_name']
batch1_2_final['local_path_2'] = PATH_FILE_SYSTEM + batch1_2_final['directory_name_2']
batch1_2_final.head(5)

,collection_name,journal_name,start_year,end_year,number_of_issues,data_transfer_batch,directory_name,directory_name_2,alias,local_path,local_path_2
0,Annuaire vaudois,Almanach pour le commerce,1832,1832,1,Batch 1 (21/12/23),Almanach_pour_le_commerce,Almanach_pour_le_commerce,ACI,/mnt/project_impresso/original/BCUL/Almanach_p...,/mnt/project_impresso/original/BCUL/Almanach_p...
1,Bulletins du Grand Conseil,Bulletins du Grand Conseil,1829,2020,619,Batch 2 (26/02/24),Bulletins_du_Grand_Conseil,Bulletins_du_Grand_Conseil,RN,/mnt/project_impresso/original/BCUL/Bulletins_...,/mnt/project_impresso/original/BCUL/Bulletins_...
2,Journaux satiriques vaudois,Castigat ridendo mores,1879,1880,6,Batch 1 (21/12/23),Castigat_ridendo_mores,Castigat_ridendo_mores,Castigat,/mnt/project_impresso/original/BCUL/Castigat_r...,/mnt/project_impresso/original/BCUL/Castigat_r...
3,Presse de l'Est vaudois,Courrier du Léman,1882,1886,445,Batch 1 (21/12/23),Courrier_du_Léman,Courrier_du_LÃ©man,CL,/mnt/project_impresso/original/BCUL/Courrier_d...,/mnt/project_impresso/original/BCUL/Courrier_d...
4,Journaux satiriques vaudois,Croquis vaudois,1884,1887,55,Batch 1 (21/12/23),Croquis_vaudois,Croquis_vaudois,Croquis,/mnt/project_impresso/original/BCUL/Croquis_va...,/mnt/project_impresso/original/BCUL/Croquis_va...


In [143]:
batch1_2_aliases_mit = batch1_2_final.copy()
batch1_2_aliases_mit['mit_file_type'] = batch1_2_final['directory_name'].apply(find_mit_extension)


In [144]:
batch1_2_aliases_mit[batch1_2_aliases_mit['mit_file_type'].isnull()]

,collection_name,journal_name,start_year,end_year,number_of_issues,data_transfer_batch,directory_name,directory_name_2,alias,local_path,local_path_2,mit_file_type
12,Annuaire vaudois,Indicateur de Lausanne,1880,1886,3,Batch 1 (21/12/23),Indicateur_de_Lausanne,Indicateur_de_Lausanne,AV,/mnt/project_impresso/original/BCUL/Indicateur...,/mnt/project_impresso/original/BCUL/Indicateur...,None
45,Almanachs vaudois,Messager boîteux,1748,1800,53,Batch 1 (21/12/23),Messager_boîteux,Messager_boÃ®teux,MB,/mnt/project_impresso/original/BCUL/Messager_b...,/mnt/project_impresso/original/BCUL/Messager_b...,None


In [145]:
# change the mit_file_type where is null to 'both'
batch1_2_aliases_mit['mit_file_type'] = batch1_2_aliases_mit['mit_file_type'].fillna('both')

In [146]:
batch1_2_aliases_mit.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   collection_name      50 non-null     object
 1   journal_name         50 non-null     object
 2   start_year           50 non-null     object
 3   end_year             50 non-null     object
 4   number_of_issues     48 non-null     object
 5   data_transfer_batch  50 non-null     object
 6   directory_name       50 non-null     object
 7   directory_name_2     50 non-null     object
 8   alias                50 non-null     object
 9   local_path           50 non-null     object
 10  local_path_2         50 non-null     object
 11  mit_file_type        50 non-null     object
dtypes: object(12)
memory usage: 4.8+ KB


In [147]:
bcul3_4_aliases.info()

<class 'pandas.core.frame.DataFrame'>
Index: 127 entries, Allez_savoir! to Yverdon_Revue__Flash_Revue
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   collection_name      127 non-null    object
 1   journal_name         127 non-null    object
 2   start_year           127 non-null    object
 3   end_year             127 non-null    object
 4   number_of_issues     127 non-null    object
 5   data_transfer_batch  127 non-null    object
 6   directory_name       127 non-null    object
 7   alias                127 non-null    object
 8   local_path           127 non-null    object
 9   mit_file_type        127 non-null    object
dtypes: object(10)
memory usage: 15.0+ KB


In [148]:
# merge both dataframes batch1_2_mit and bcul3_4_aliases and create final_batches
final_batches = pd.concat([batch1_2_aliases_mit, bcul3_4_aliases], ignore_index=True)
final_batches

,collection_name,journal_name,start_year,end_year,number_of_issues,data_transfer_batch,directory_name,directory_name_2,alias,local_path,local_path_2,mit_file_type
0,Annuaire vaudois,Almanach pour le commerce,1832,1832,1,Batch 1 (21/12/23),Almanach_pour_le_commerce,Almanach_pour_le_commerce,ACI,/mnt/project_impresso/original/BCUL/Almanach_p...,/mnt/project_impresso/original/BCUL/Almanach_p...,xml
1,Bulletins du Grand Conseil,Bulletins du Grand Conseil,1829,2020,619,Batch 2 (26/02/24),Bulletins_du_Grand_Conseil,Bulletins_du_Grand_Conseil,RN,/mnt/project_impresso/original/BCUL/Bulletins_...,/mnt/project_impresso/original/BCUL/Bulletins_...,json
2,Journaux satiriques vaudois,Castigat ridendo mores,1879,1880,6,Batch 1 (21/12/23),Castigat_ridendo_mores,Castigat_ridendo_mores,Castigat,/mnt/project_impresso/original/BCUL/Castigat_r...,/mnt/project_impresso/original/BCUL/Castigat_r...,xml
3,Presse de l'Est vaudois,Courrier du Léman,1882,1886,445,Batch 1 (21/12/23),Courrier_du_Léman,Courrier_du_LÃ©man,CL,/mnt/project_impresso/original/BCUL/Courrier_d...,/mnt/project_impresso/original/BCUL/Courrier_d...,json
4,Journaux satiriques vaudois,Croquis vaudois,1884,1887,55,Batch 1 (21/12/23),Croquis_vaudois,Croquis_vaudois,Croquis,/mnt/project_impresso/original/BCUL/Croquis_va...,/mnt/project_impresso/original/BCUL/Croquis_va...,xml
...,...,...,...,...,...,...,...,...,...,...,...,...
172,Presse de la Côte vaudoise,Reflets de la Côte,1980,1982,26,Batch 3 (12/24),Reflets_de_la_Cote,NaN,REC,/mnt/project_impresso/original/BCUL/Reflets_de...,NaN,json
173,Presse de la Côte vaudoise,Revue morgienne,1902,1905,33,Batch 3 (12/24),Revue_morgienne,NaN,RMorg,/mnt/project_impresso/original/BCUL/Revue_morg...,NaN,json
174,Journaux satiriques vaudois,Revue photo-rigolotipique,1902,1902,1,Batch 3 (12/24),Revue_photo-rigolotipique,NaN,RPR,/mnt/project_impresso/original/BCUL/Revue_phot...,NaN,xml
175,"Presse de la Broye, du Jorat et du Gros-de-Vaud",Terre vuilleraine,1953,2010,2579,Bath 4 (08/25),Terre_vuilleraine,NaN,TV,/mnt/project_impresso/original/BCUL/Terre_vuil...,NaN,json


In [149]:
final_batches.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 177 entries, 0 to 176
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   collection_name      177 non-null    object
 1   journal_name         177 non-null    object
 2   start_year           177 non-null    object
 3   end_year             177 non-null    object
 4   number_of_issues     175 non-null    object
 5   data_transfer_batch  177 non-null    object
 6   directory_name       177 non-null    object
 7   directory_name_2     50 non-null     object
 8   alias                177 non-null    object
 9   local_path           177 non-null    object
 10  local_path_2         50 non-null     object
 11  mit_file_type        177 non-null    object
dtypes: object(12)
memory usage: 16.7+ KB


In [ ]:
batch1_2_aliases_mit['mit_file_type'] = batch1_2_aliases_mit['mit_file_type'].fillna('both')

In [154]:
final_batches['directory_name_2'] = final_batches['directory_name_2'].fillna(final_batches['directory_name'])
final_batches['local_path_2'] = final_batches['local_path_2'].fillna(final_batches['local_path'])

In [155]:
final_batches.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 177 entries, 0 to 176
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   collection_name      177 non-null    object
 1   journal_name         177 non-null    object
 2   start_year           177 non-null    object
 3   end_year             177 non-null    object
 4   number_of_issues     175 non-null    object
 5   data_transfer_batch  177 non-null    object
 6   directory_name       177 non-null    object
 7   directory_name_2     177 non-null    object
 8   alias                177 non-null    object
 9   local_path           177 non-null    object
 10  local_path_2         177 non-null    object
 11  mit_file_type        177 non-null    object
dtypes: object(12)
memory usage: 16.7+ KB


In [156]:
# build the dictionary
data_dict = {
    row["directory_name"]: row.to_dict()
    for _, row in final_batches.iterrows()
}

# convert to JSON (pretty-printed)
json_output = json.dumps(data_dict, indent=4, ensure_ascii=False)

# optional: save to file
with open("/rcp-scratch/iccluster040_scratch/students/banuls/impresso-text-acquisition/text_preparation/data/sample_data/BCUL/bcul_aliases.json", "w", encoding="utf-8") as f:
    f.write(json_output)

print(json_output)

{
    "Almanach_pour_le_commerce": {
        "collection_name": "Annuaire vaudois",
        "journal_name": "Almanach pour le commerce",
        "start_year": "1832",
        "end_year": "1832",
        "number_of_issues": "1",
        "data_transfer_batch": "Batch 1 (21/12/23)",
        "directory_name": "Almanach_pour_le_commerce",
        "directory_name_2": "Almanach_pour_le_commerce",
        "alias": "ACI",
        "local_path": "/mnt/project_impresso/original/BCUL/Almanach_pour_le_commerce",
        "local_path_2": "/mnt/project_impresso/original/BCUL/Almanach_pour_le_commerce",
        "mit_file_type": "xml"
    },
    "Bulletins_du_Grand_Conseil": {
        "collection_name": "Bulletins du Grand Conseil",
        "journal_name": "Bulletins du Grand Conseil",
        "start_year": "1829",
        "end_year": "2020",
        "number_of_issues": "619",
        "data_transfer_batch": "Batch 2 (26/02/24)",
        "directory_name": "Bulletins_du_Grand_Conseil",
        "directory_n

In [97]:
# load bcul_aliases3_4.json
temp = pd.read_json('/rcp-scratch/iccluster040_scratch/students/banuls/impresso-text-acquisition/text_preparation/data/sample_data/BCUL/bcul_aliases.json').T
temp.head(2)

,collection_name,journal_name,start_year,end_year,number_of_issues,data_transfer_batch,directory_name,bin_title,alias,local_path,mit_file_type
Almanach_pour_le_commerce,Annuaire vaudois,Almanach pour le commerce,1832,1832,1,Batch 1 (21/12/23),Almanach_pour_le_commerce,Almanach_pour_le_commerce,ACI,/mnt/project_impresso/original/BCUL/Almanach_p...,xml
Bulletins_du_Grand_Conseil,Bulletins du Grand Conseil,Bulletins du Grand Conseil,1829,2020,619,Batch 2 (26/02/24),Bulletins_du_Grand_Conseil,None,RN,/mnt/project_impresso/original/BCUL/Bulletins_...,json
